# Pre-Release Quarterback Pocket Context

Does observable quarterback movement and pass-rush proximity before release help predict receiver separation creation? The target remains `delta_sep = sep_release - sep_snap`. We compare the existing static ridge model, its defender-motion extension, and a third ridge model that adds pocket-context features. All models use the same five game-grouped folds stratified within week; every route is scored out of fold.

The new inputs summarize QB movement and PFF-tagged pass-rusher distance from snap up to, but excluding, the release frame. They are contextual predictors, not an attribution of pressure or route quality.

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
from separation_over_expected.reports import read_csv_rows

EXPERIMENT_DIR = PROJECT_ROOT / "data" / "processed" / "dynamic_features" / "pocket_context"
ROUTES = read_csv_rows(EXPERIMENT_DIR / "route_level_snap_to_release_pocket.csv")
METRICS = read_csv_rows(EXPERIMENT_DIR / "cross_validation" / "oof_metrics_wr.csv")
RELIABILITY = read_csv_rows(EXPERIMENT_DIR / "cross_validation" / "receiver_oof_reliability_wr.csv")
len(ROUTES), len(METRICS), len(RELIABILITY)

(35032, 18, 5)

## Feature coverage and range checks

The route table includes one row per route runner on eligible dropbacks. Check that pocket features are populated and inspect their broad distributions before interpreting model scores.

In [4]:
POCKET_FEATURES = [
    "qb_depth_drop_pre_release", "qb_lateral_drift_pre_release", "qb_path_length_pre_release",
    "qb_mean_speed_pre_release", "qb_max_speed_pre_release", "qb_mean_accel_pre_release",
    "qb_nearest_rusher_dist_snap", "qb_nearest_rusher_min_dist_pre_release",
    "qb_nearest_rusher_mean_dist_pre_release", "qb_rusher_closing_rate_pre_release",
    "qb_rusher_within_3yd_frame_share_pre_release", "qb_rusher_within_5yd_frame_share_pre_release",
    "qb_pressure_observed_fraction_pre_release", "pff_pass_rusher_count",
]
def feature_summary(rows, name):
    values = [float(row[name]) for row in rows if row.get(name, "") != ""]
    return {"feature": name, "missing": len(rows) - len(values), "min": round(min(values), 3),
            "mean": round(sum(values) / len(values), 3), "max": round(max(values), 3)}

coverage = [feature_summary(ROUTES, name) for name in POCKET_FEATURES]
coverage

[{'feature': 'qb_depth_drop_pre_release', 'missing': 0, 'min': -17.01, 'mean': -3.429, 'max': 5.17}, {'feature': 'qb_lateral_drift_pre_release', 'missing': 0, 'min': -28.91, 'mean': -0.401, 'max': 28.2}, {'feature': 'qb_path_length_pre_release', 'missing': 0, 'min': 0.121, 'mean': 5.596, 'max': 43.376}, {'feature': 'qb_mean_speed_pre_release', 'missing': 0, 'min': 0.106, 'mean': 1.741, 'max': 6.12}, {'feature': 'qb_max_speed_pre_release', 'missing': 0, 'min': 0.24, 'mean': 3.301, 'max': 9.45}, {'feature': 'qb_mean_accel_pre_release', 'missing': 0, 'min': 0.175, 'mean': 1.925, 'max': 4.78}, {'feature': 'qb_nearest_rusher_dist_snap', 'missing': 4, 'min': 1.569, 'mean': 5.305, 'max': 8.336}, {'feature': 'qb_nearest_rusher_min_dist_pre_release', 'missing': 4, 'min': 0.04, 'mean': 2.805, 'max': 7.547}, {'feature': 'qb_nearest_rusher_mean_dist_pre_release', 'missing': 4, 'min': 1.958, 'mean': 4.574, 'max': 9.406}, {'feature': 'qb_rusher_closing_rate_pre_release', 'missing': 4, 'min': -3.591,

## Out-of-fold route prediction

All three models score the same 20,415 WR routes. The paired model comparisons are the key: pocket features must improve on the existing dynamic model to add value beyond defender motion already in the pipeline.

In [6]:
overall = [row for row in METRICS if row["fold"] == "OOF_ALL"]
overall

[{'fold': 'OOF_ALL', 'model': 'ridge_context', 'mae': '1.390', 'rmse': '1.897', 'r2': '0.493', 'bias': '0.001', 'mean_actual': '-2.341', 'mean_predicted': '-2.341', 'n': '20415'}, {'fold': 'OOF_ALL', 'model': 'ridge_dynamic_context', 'mae': '1.363', 'rmse': '1.869', 'r2': '0.508', 'bias': '0.001', 'mean_actual': '-2.341', 'mean_predicted': '-2.342', 'n': '20415'}, {'fold': 'OOF_ALL', 'model': 'ridge_pocket_context', 'mae': '1.362', 'rmse': '1.867', 'r2': '0.509', 'bias': '0.001', 'mean_actual': '-2.341', 'mean_predicted': '-2.341', 'n': '20415'}]

In [7]:
fold_results = [row for row in METRICS if row["fold"] != "OOF_ALL"]
fold_results

[{'fold': '1', 'model': 'ridge_context', 'mae': '1.363', 'rmse': '1.853', 'r2': '0.496', 'bias': '0.011', 'mean_actual': '-2.276', 'mean_predicted': '-2.287', 'n': '4972'}, {'fold': '1', 'model': 'ridge_dynamic_context', 'mae': '1.335', 'rmse': '1.826', 'r2': '0.511', 'bias': '0.011', 'mean_actual': '-2.276', 'mean_predicted': '-2.287', 'n': '4972'}, {'fold': '1', 'model': 'ridge_pocket_context', 'mae': '1.334', 'rmse': '1.823', 'r2': '0.513', 'bias': '0.007', 'mean_actual': '-2.276', 'mean_predicted': '-2.283', 'n': '4972'}, {'fold': '2', 'model': 'ridge_context', 'mae': '1.409', 'rmse': '1.906', 'r2': '0.503', 'bias': '-0.096', 'mean_actual': '-2.583', 'mean_predicted': '-2.487', 'n': '3942'}, {'fold': '2', 'model': 'ridge_dynamic_context', 'mae': '1.377', 'rmse': '1.875', 'r2': '0.519', 'bias': '-0.084', 'mean_actual': '-2.583', 'mean_predicted': '-2.499', 'n': '3942'}, {'fold': '2', 'model': 'ridge_pocket_context', 'mae': '1.375', 'rmse': '1.871', 'r2': '0.521', 'bias': '-0.088', '

## Receiver-level reliability

We split the out-of-fold predictions into two disjoint, week-balanced game sets and correlate each receiver's average SOE between halves. The paired bootstrap resamples the same 115 receivers for both models. Route prediction and receiver reliability answer different questions: a small average route error gain need not mean player evaluations become more repeatable.

In [9]:
RELIABILITY

[{'comparison': 'ridge_context', 'eligible_receivers': '115', 'pearson': '0.430880', 'spearman': '0.220556', 'pearson_ci_lower': '0.071785', 'pearson_ci_upper': '0.672905', 'spearman_ci_lower': '0.032200', 'spearman_ci_upper': '0.386949', 'minimum_routes_each_half': '20'}, {'comparison': 'ridge_dynamic_context', 'eligible_receivers': '115', 'pearson': '0.451857', 'spearman': '0.209469', 'pearson_ci_lower': '0.079452', 'pearson_ci_upper': '0.691667', 'spearman_ci_lower': '0.022494', 'spearman_ci_upper': '0.384713', 'minimum_routes_each_half': '20'}, {'comparison': 'ridge_pocket_context', 'eligible_receivers': '115', 'pearson': '0.423567', 'spearman': '0.184195', 'pearson_ci_lower': '0.046663', 'pearson_ci_upper': '0.671889', 'spearman_ci_lower': '-0.006326', 'spearman_ci_upper': '0.361639', 'minimum_routes_each_half': '20'}, {'comparison': 'dynamic_minus_static', 'eligible_receivers': '115', 'pearson': '0.020977', 'spearman': '-0.011087', 'pearson_ci_lower': '-0.032642', 'pearson_ci_upp

## Takeaway

Pocket context gives a tiny additional route-level gain over dynamic defender context: OOF RMSE moves from 1.869 to 1.867 and R² from 0.508 to 0.509. The direction of the RMSE change is consistent across most folds, but its magnitude is negligible.

Receiver reliability moves the other way. Split-half Pearson correlation is 0.424 for the pocket model versus 0.452 for dynamic context; the paired difference is -0.028 (95% bootstrap interval -0.050 to -0.011). Spearman ranking correlation also falls from 0.209 to 0.184, though the paired interval includes zero. On this evaluation, the new features do not improve receiver evaluation and may make level estimates less repeatable.

Recommendation: keep the pocket features as an experiment, not as the preferred model inputs. The route-level improvement is too small to justify the reduction in Pearson receiver stability. Before trying a more complex model, inspect the added feature coefficients/regularization and test a narrower pressure feature set (for example, rusher proximity and closing only) against the exact same folds. This is a within-season, eight-week result; it does not establish year-to-year stability.